# Évaluation finale : Adult Income (UCI)

Ce notebook réalise l'évaluation finale sur le jeu de test `adult.test`.
Le modèle retenu, ses hyperparamètres et le seuil de décision proviennent de `hyperopt.ipynb`
(via `artifacts/final_model.json`)


In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import classification_report

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from adult_income import DataLoader, PreprocessingPipeline, ModelRegistry, ModelEvaluator

print("Imports OK")

## 1. Chargement du modèle final & des données

On recharge le modèle retenu (`artifacts/final_model.json`), on réentraîne sur tout le
train (`adult.data`) puis on prédit sur `adult.test`. Le prétraitement reproduit le choix
d'ablation (`capital_treatment="naive"`)


In [ ]:
spec = json.loads((ROOT / "artifacts" / "final_model.json").read_text())
model_name = spec["model_name"]
best_params = spec["best_params"]
threshold = float(spec["threshold"])
print(f"Modèle final : {model_name} | seuil retenu : {threshold:.3f}")
print(f"Hyperparamètres : {best_params}")

loader = DataLoader()
X_train, y_train = loader.load_train()
X_test, y_test = loader.load_test()

pp = PreprocessingPipeline(capital_treatment="naive")
y_train_enc = pp.fit_transform_target(y_train)
y_test_enc = pp.transform_target(y_test)

est = ModelRegistry.get(model_name)
est.set_params(**best_params)
final_pipeline = pp.build_pipeline(est)

evaluator = ModelEvaluator()
evaluator.evaluate_on_test(model_name, final_pipeline, X_train, y_train_enc, X_test, y_test_enc)
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= threshold).astype(int)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

## 2. Comparaison des seuils (0,50 vs seuil retenu)

In [ ]:
evaluator.threshold_comparison(y_test_enc, y_test_proba, [0.50, threshold]).round(4)

## 3. Rapport de classification (au seuil retenu)

In [ ]:
print(classification_report(y_test_enc, y_test_pred, target_names=["<=50K", ">50K"], zero_division=0))

## 4. Matrices de confusion

In [ ]:
evaluator.plot_confusion_pair(y_test_enc, y_test_pred)
plt.show()

## 5. Courbes ROC et précision-rappel

La ligne de base PR correspond à la prévalence de `>50K` (~24 %).


In [ ]:
evaluator.plot_roc_curves(X_test, y_test_enc)
plt.show()
evaluator.plot_pr_curves(X_test, y_test_enc)
plt.show()

## 6. Analyse des erreurs

Décomposition des prédictions (au seuil retenu) en VP / VN / FP / FN, puis profil numérique
moyen de chaque type.


In [ ]:
profil_cols = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss"]
counts, profiles = evaluator.error_breakdown(X_test, y_test_enc, y_test_pred, y_test_proba, profil_cols)
display(counts)
profiles

## 7. Performances selon le sexe

Précision / rappel / F1 et taux de faux positifs (FPR) / faux négatifs (FNR) par groupe


In [ ]:
evaluator.group_metrics("sex").round(4)

## 8. Performances selon l'origine (`race`)

Certains groupes possèdent un effectif très faible, donc pas vraiment représentatif de la réalité, il ne faut pas tirer des conclusions trop hatives sur cette analyse


In [ ]:
evaluator.group_metrics("race").round(4)

## 9. Importance des variables

Pour les modèles à arbres/boosting, importance native sur les colonnes issues du
prétraitement (post one-hot)


In [ ]:
fig = evaluator.plot_feature_importance(final_pipeline, top_n=20)
if fig is not None:
    plt.show()
    display(evaluator.feature_importance(final_pipeline, top_n=20))
else:
    print(f"{model_name} n'expose pas d'importance native des variables.")

## 10. Test de sensibilité sans `sex` ni `race`

La pipeline de prétraitement référence `sex` et `race` en dur (groupements EDA). Plutôt que de
reconstruire un transformateur, on neutralise ces deux variables en les remplaçant
par leur modalité majoritaire sur le train et le test : elles ne portent alors plus aucune
information. On réentraîne et on compare au modèle complet


In [ ]:
X_train_ns, X_test_ns = X_train.copy(), X_test.copy()
for col in ["sex", "race"]:
    const = X_train[col].mode()[0]
    X_train_ns[col] = const
    X_test_ns[col] = const

pipe_ns = clone(final_pipeline)
pipe_ns.fit(X_train_ns, y_train_enc)
proba_ns = pipe_ns.predict_proba(X_test_ns)[:, 1]

pd.DataFrame([
    {"modèle": "avec sex + race", **evaluator.metrics_at_threshold(y_test_enc, y_test_proba, threshold)},
    {"modèle": "sans sex + race (neutralisés)", **evaluator.metrics_at_threshold(y_test_enc, proba_ns, threshold)},
]).round(4)

## 11. Lecture non-supervisée du test : t-SNE & k-means

On réutilise ici les outils non-supervisés de l'AED (`unsupervised.ipynb`), mais cette fois sur le jeu de test et en les croisant avec les sorties du modèle. L'objectif est d'essayer de voir où le modèle se trompe dans la géométrie des données, et non plus seulement combien d'erreurs on obtient. Rappel : t-SNE et k-means restent "qualitatifs" dans le sens ou ils servent à faire des observations et pas à tirer des conclusions chiffrées (distances et tailles de clusters non fiables, espace one-hot)

In [ ]:
import numpy as np
from adult_income import ClusteringAnalyzer

ca_test = ClusteringAnalyzer(X_test, y_test, random_state=42)

erreurs = (np.asarray(y_test_pred) != np.asarray(y_test_enc)).astype(int)
print(f"Test : {len(X_test):,} obs | taux d'erreur global : {erreurs.mean():.1%}")

### 11.1. k-means sur le test

Choix de `k` via coude + silhouette, puis profil de chaque cluster enrichi du taux d'erreur du modèle. Un cluster « frontière » (taux de >50K proche de 50 %) qui concentre les erreurs confirmerait que le modèle bute sur le chevauchement des classes, pas sur un défaut global. Permet de voir si les erreurs se situent dans un cluster en particulier.

Lien avec l'homogénéité et la séparabilité : l'inertie du coude mesure à quel point les points sont serrés à l'intérieur des clusters, c'est donc l'homogénéité (et c'est ce que k-means cherche à minimiser). La séparabilité, c'est au contraire à quel point les clusters sont éloignés les uns des autres. La silhouette regroupe ces deux idées en un seul score, ce qui explique son maximum à k=2.

In [ ]:
ca_test.plot_kmeans_search(k_values=range(2, 11))
plt.show()

In [ ]:
# k retenu d'après le coude / la silhouette ci-dessus. k=4 donne des profils plus fins que la partition silhouette-optimale (k=2)
ca_test.fit_kmeans(k=4)

profil_cols = ["age", "education-num", "hours-per-week"]
ca_test.cluster_profile(
    numeric=profil_cols,
    extra={"taux_err_modèle (%)": erreurs * 100},
).round(2)

### 11.2. t-SNE du test : vrai / prédit / erreurs côte à côte

Trois cartes sur le même embedding t-SNE : vraies classes, classes prédites, puis erreurs.

On voit que les erreurs (3ème plot) se concentrent en haut, là où il y a beaucoup de >50K (rouge) mélangés aux <=50K dans le 1er plot. Le gros bloc du bas, presque uniquement composé de <=50K, ne contient quasiment pas d'erreurs. Autrement dit, le modèle se trompe surtout là où les deux classes se chevauchent (les profils sont ambigus), pas un peu partout au hasard : ses erreurs sont donc structurelles, et portent surtout sur la classe >50K, ce qui rejoint son rappel plus faible sur >50K vu en supervisé.

In [ ]:
# Même embedding réutilisé pour les trois plots (cache interne de ca_test)
fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))

ca_test.plot_tsne(color_by="income", ax=axes[0], title="Vraies classes")

ca_test.plot_tsne(
    color_by="custom", values=np.asarray(y_test_pred), ax=axes[1],
    value_labels={0: "<=50K", 1: ">50K"},
    value_colors={0: "steelblue", 1: "tomato"},
    legend_title="prédiction", title="Classes prédites",
)

ca_test.plot_tsne(
    color_by="custom", values=erreurs, ax=axes[2],
    value_labels={0: "bien classé", 1: "mal classé"},
    value_colors={0: "lightgrey", 1: "crimson"},
    legend_title="modèle", title="Erreurs de classification",
)
plt.show()

## 12. Discussion & recommandation

In [ ]:
recap = evaluator.metrics_at_threshold(y_test_enc, y_test_proba, threshold)
print(f"Modèle recommandé : {model_name}")
print(f"Seuil de décision : {threshold:.3f}")
print(f"Test  : Average Precision : {recap['average_precision']:.4f} | ROC-AUC : {recap['roc_auc']:.4f}")
print(f"Test  : F1 : {recap['f1']:.4f} | précision : {recap['precision']:.4f} | rappel : {recap['recall']:.4f}")

**Synthèse :** Le modèle retenu est sélectionné sur l'**Average Precision**
en validation croisée, métrique adaptée au déséquilibre (~24 % de `>50K`). Le réglage du
seuil via validation croisée également déplace le compromis vers un meilleur **F1** que le seuil par
défaut de 0,50.

**Apport du seuil :** La comparaison vue en section 2 montre l'effet du seuil retenu vs 0,50 (typiquement
plus de rappel sur la classe minoritaire au prix d'un peu de précision).

**Équité :** Les écarts de performance par `sex` / `race` (sections 7 et 8) sont à lire avec prudence
(faibles effectifs sur certains groupes). Le test de sensibilité  de la section 10 indique que retirer
`sex`/`race` ne change que marginalement les performances : ces variables apportent peu de
gain prédictif direct, mais leur retrait ne garantit pas l'absence de biais (possible que d'autres variables soient porteuses de ces biais).

**Limites :** Données de recensement 1994 (non représentatives d'aujourd'hui), top-coding
de `capital-gain` (99999) et `hours-per-week` (99), corrélations socio-démographiques fortes.

**Recommandation :** Retenir le modèle ci-dessus au seuil réglé pour un usage privilégiant le
rappel sur `>50K`, et conserver 0,50 si l'on préfère maximiser la précision. Surveiller l'équité
en production et réentraîner sur des données récentes